In [2]:
from amplpy import AMPL
import pandas as pd
import time

In [3]:
# -----------------------------
# 1. Load Excel Price Data
# -----------------------------
# Ensure Prices.xlsx exists with column "NP15 ($/MWh)"
df_prices = pd.read_excel("Prices.xlsx")
df_prices["t"] = range(1, len(df_prices) + 1)
prices = df_prices.set_index("t")["NP15 ($/MWh)"]
T = df_prices["t"].tolist()

In [4]:
ampl = AMPL()

In [5]:
# -----------------------------
# 2. Define Model
# -----------------------------
ampl.eval("""
set T ordered;
set C;
set S;

param price{T};
param gas_price;
param CO2Cost;

param Pmin{C};
param Pmax{C};
param SU{C};
param SUF{C};
param VOM{C};

param seg_max{C,S} default 0;
param HR{C,S} default 0;

var u{C,T} binary;
var y{C,T} binary;
var p{C,T} >= 0;
var ps{C,S,T} >= 0;

maximize Profit:
    sum{t in T} (
        price[t] * sum{c in C} p[c,t]
      - sum{c in C, s in S} (HR[c,s]*gas_price + VOM[c]) * ps[c,s,t]
      - sum{c in C} y[c,t]*(SU[c] + gas_price*SUF[c])
    );

s.t. OneConfig{t in T}:
    sum{c in C} u[c,t] = 1;

s.t. GenMin{c in C, t in T}:
    p[c,t] >= Pmin[c]*u[c,t];

s.t. GenMax{c in C, t in T}:
    p[c,t] <= Pmax[c]*u[c,t];

s.t. Link{c in C, t in T}:
    p[c,t] = sum{s in S} ps[c,s,t];

s.t. SegLimit{c in C, s in S, t in T}:
    ps[c,s,t] <= seg_max[c,s]*u[c,t];

s.t. Startup{c in C, t in T: ord(t) > 1}:
    y[c,t] >= u[c,t] - u[c,t-1];

s.t. Transition{t in T: ord(t) > 1}:
    y[5,t] <= u[4,t-1];
""")

In [6]:
# -----------------------------
# 3. Define Sets
# -----------------------------
C = [1, 2, 3, 4, 5]
S = [1, 2, 3]
ampl.set["T"] = T
ampl.set["C"] = C
ampl.set["S"] = S

In [7]:
# -----------------------------
# 4. Parameters (Scalars & 1D)
# -----------------------------
ampl.param["price"] = prices
ampl.param["gas_price"] = 7
ampl.param["CO2Cost"] = 15

# Using Series for 1D parameters (indexed by C)
config_data = pd.DataFrame({
    "Pmin": [0, 57, 150, 150, 312],
    "Pmax": [0, 190, 340, 340, 610],
    "VOM":  [0, 5.0, 2.5, 2.5, 2.0],
    "SU":   [0, 7250, 14500, 16500, 23750],
    "SUF":  [0, 220, 440, 850, 1070]
}, index=C)

# Bulk load 1D parameters
for col in config_data.columns:
    ampl.param[col] = config_data[col]

In [8]:
# -----------------------------
# 5. Parameters (Multi-Indexed 2D) - FIXED
# -----------------------------
seg_hr_data = {
    (1,1): [0, 0], (1,2): [0, 0], (1,3): [0, 0],
    (2,1): [57, 8.692], (2,2): [38, 9.177], (2,3): [38, 9.564],
    (3,1): [54, 8.692], (3,2): [68, 9.177], (3,3): [68, 9.564],
    (4,1): [54, 6.421], (4,2): [68, 6.525], (4,3): [68, 6.640],
    (5,1): [54, 6.292], (5,2): [122, 6.361], (5,3): [122, 6.452]
}

# 1. Create the DataFrame
df_seg = pd.DataFrame.from_dict(seg_hr_data, orient='index', columns=['seg_max', 'HR'])

# 2. Convert the index from Tuples to a MultiIndex
df_seg.index = pd.MultiIndex.from_tuples(df_seg.index, names=['C', 'S'])

# 3. Load into AMPL
ampl.param["seg_max"] = df_seg["seg_max"]
ampl.param["HR"] = df_seg["HR"]

In [ ]:
# -----------------------------
# 6. Initial Conditions & Solve (Optimized for Jupyter/IPython)
# -----------------------------
initial_config = 1
for c in C:
    val = 1 if c == initial_config else 0
    ampl.getVariable("u")[c, 1].fix(val)

# Use a simpler options string and clear existing ones
ampl.set_option("solver", "gurobi")
ampl.set_option("gurobi_options", "outlev=1 mipgap=0.01 threads=8 timing=1")

print("Starting Solve...")
# Solving with 'solve' can sometimes hang in Notebooks if the output buffer is flooded
# We use 'solve' but add a print right after to confirm return
ampl.eval("solve;");
print("Solver returned control to Python.") 

# Force a value refresh
total_profit = ampl.get_objective("Profit").value()
print(f"Profit: ${total_profit:,.2f}")

Starting Solve...
Gurobi 13.0.0: Set parameter LogToConsole to value 1
  tech:outlev = 1
Set parameter MIPGap to value 0.01
  mip:gap = 0.01
Set parameter Threads to value 8
  tech:threads = 8
  tech:timing = 1

AMPL MP initial flat model has 3509 variables (0 integer, 1505 binary);
Objectives: 1 linear; 
Constraints:  4006 linear;

AMPL MP final model has 3509 variables (0 integer, 1505 binary);
Objectives: 1 linear; 
Constraints:  4006 linear;


Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Snapdragon X Plus (8-core) @ 3.30 GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0.01
InfUnbdInfo  1
Threads  8

Optimize a model with 4006 rows, 3509 columns and 10012 nonzeros (Max)
Model fingerprint: 0xe069afd0
Model has 2675 linear objective coefficients
Variable types: 2004 continuous, 1505 integer (0 binary)
C

In [ ]:
# -----------------------------
# 7. Results Retrieval
# -----------------------------
total_profit = ampl.get_objective("Profit").value()
print(f"\nOptimization Complete in {end_time - start_time:.2f}s")
print(f"Total Projected Profit: ${total_profit:,.2f}")

# Get variable values as DataFrames
u_results = ampl.getVariable("u").getValues().toPandas()
p_results = ampl.getVariable("p").getValues().toPandas()

# Example: Show hours where the plant is generating
active_gen = p_results[p_results['val'] > 0.1]
print("\nActive Generation Schedule (First 5 Rows):")
print(active_gen.head())